## wikipedia Passages vectorization

In [ ]:
import json
import time
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/all-mpnet-base-v2",
    device="cuda"
).half()

INPUT_FILE = "all_unique_passages.json"
OUTPUT_FILE = "data_with_embeddings.jsonl"

BATCH_SIZE = 12000
ENCODE_BATCH = 256


def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]


with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

out = open(OUTPUT_FILE, "a", encoding="utf-8")

for batch_num, batch in enumerate(chunks(data, BATCH_SIZE), 1):
    start = time.time()

    texts = [item["text"] for item in batch]

    embeddings = model.encode(
        texts,
        batch_size=ENCODE_BATCH,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False
    )

    for item, emb in zip(batch, embeddings):
        item["embed_text"] = emb.tolist()
        out.write(json.dumps(item, ensure_ascii=False) + "\n")

    out.flush()

    print(f"Batch {batch_num} done in {time.time()-start:.2f}s")

out.close()

d:\git code\Reranker\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\git code\Reranker\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Batch 1 done in 217.68s
Batch 2 done in 35.76s
Batch 3 done in 16.44s
Batch 4 done in 16.06s
Batch 5 done in 4.48s
Batch 6 done in 10.45s
Batch 7 done in 11.91s
Batch 8 done in 4.46s
Batch 9 done in 101.25s
Batch 10 done in 18.44s
Batch 11 done in 28.16s
Batch 12 done in 143.16s
Batch 13 done in 12.38s
Batch 14 done in 50.34s
Batch 15 done in 4.82s
Batch 16 done in 13.26s
Batch 17 done in 5.05s
Batch 18 done in 121.03s
Batch 19 done in 34.54s
Batch 20 done in 106.56s
Batch 21 done in 41.76s
Batch 22 done in 18.86s
Batch 23 done in 23.44s
Batch 24 done in 11.18s
Batch 25 done in 19.24s
Batch 26 done in 90.35s
Batch 27 done in 5.77s
Batch 28 done in 106.65s
Batch 29 done in 5.95s
Batch 30 done in 6.34s
Batch 31 done in 89.95s
Batch 32 done in 5.06s
Batch 33 done in 17.74s
Batch 34 done in 15.33s
Batch 35 done in 4.96s
Batch 36 done in 24.78s
Batch 37 done in 15.72s
Batch 38 done in 18.79s
Batch 39 done in 28.08s
Batch 40 done in 20.37s
Batch 41 done in 32.36s
Batch 42 done in 10.05s
Batc

In [ ]:
import json
import faiss
import numpy as np
import pickle

INPUT_JSONL = "data_with_embeddings.jsonl"
FAISS_INDEX_PATH = "passage_index.faiss"
MAPPING_PATH = "passage_mapping.pkl"

# Count total
with open(INPUT_JSONL, "r", encoding="utf-8") as f:
    total = sum(1 for _ in f)

# Read JSONL into numpy array
DIMENSION = None
embeddings = None
passage_ids = []
passage_texts = []

with open(INPUT_JSONL, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        data = json.loads(line)
        emb = data["embed_text"]

        if embeddings is None:
            DIMENSION = len(emb)
            embeddings = np.empty((total, DIMENSION), dtype="float32")

        embeddings[i] = emb
        passage_ids.append(data["passage_id"])
        passage_texts.append({"title": data["title"], "text": data["text"]})

# Normalize
faiss.normalize_L2(embeddings)

# Build index in batches
index = faiss.IndexFlatIP(DIMENSION)

BATCH_SIZE = 50_000
for start in range(0, total, BATCH_SIZE):
    index.add(embeddings[start : start + BATCH_SIZE])

print("Total vectors indexed:", index.ntotal)

# Save
faiss.write_index(index, FAISS_INDEX_PATH)

with open(MAPPING_PATH, "wb") as f:
    pickle.dump({"passage_ids": passage_ids, "passages": passage_texts}, f)

Total vectors indexed: 5000000


## Data set Creation for MLP model Training

In [ ]:
import json
def read_json(path):
    qa_data = []
    f = open(path, 'r', encoding='utf-8')
    for line in f.readlines():
        qa_data.append(json.loads(line))
    return qa_data


def write_jsonl(data, path):
    with open(path, 'w') as f:
        for item in data:
            f.write(json.dumps(item) + "\n")
    f.close()

def generate_rag_prompt(question: str, context: str) -> str:
    prompt = f"""
                **System Role:**
                You are a rigorous language model. Please answer the question based on the provided context. 
                If the context does not support reasoning about the answer, please answer the question based on your own knowledge.
                **Context:**
                {context}
                **Question:** 
                {question}
            """.strip()
    return prompt

def create_nq_rag_test_convert(input_jsonl, output_jsonl):
    meta_data = read_json(input_jsonl)
    result = []
    question_id = 0
    for meta in meta_data:
        context_list = meta['context']
        # Add raw question
        result.append({
            "question": meta['question'],
            "question_id": question_id,
            "context": ""
        })
        # Add one entry per context
        for context in context_list:
            question_prompt = generate_rag_prompt(meta['question'], context)
            result.append({
                "question": question_prompt,
                "question_id": question_id,
                "context": context
            })
        question_id += 1
    write_jsonl(result, output_jsonl)

create_nq_rag_test_convert("nq_questions_with_context2.jsonl", "nq_rag_test_convert2.jsonl")

In [ ]:
import json
import faiss
import pickle
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

# -----------------------------
# CONFIG
# -----------------------------
NQ_INPUT = "data\\nq-dev.jsonl"
OUTPUT_FILE = "nq_with_top5_contexts.jsonl"
FAISS_INDEX_PATH = "passage_index.faiss"
MAPPING_PATH = "passage_mapping.pkl"
TOP_K = 12
EMBED_MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"
BATCH_SIZE = 64 


# -----------------------------
# 1. Load FAISS index
# -----------------------------
index = faiss.read_index(FAISS_INDEX_PATH)

with open(MAPPING_PATH, "rb") as f:
    mapping = pickle.load(f)

passage_texts = mapping["passages"]   
del mapping                           


# -----------------------------
# 2. Load embedding model
# -----------------------------
model = SentenceTransformer(EMBED_MODEL_NAME)


# -----------------------------
# 3. Read all questions
# -----------------------------
all_data = []
with open(NQ_INPUT, "r", encoding="utf-8") as fin:
    for line in fin:
        d = json.loads(line)
        all_data.append({
            "question": d["question"],
            "reference": d.get("reference", [])
        })

print(f"Total questions: {len(all_data)}")


# -----------------------------
# 4. Batch encode + search
# -----------------------------

with open(OUTPUT_FILE, "w", encoding="utf-8") as fout:
    for batch_start in tqdm(range(0, len(all_data), BATCH_SIZE)):
        batch = all_data[batch_start : batch_start + BATCH_SIZE]
        questions = [item["question"] for item in batch]

        # ── Batch encode  ──
        query_embeddings = model.encode(
            questions,
            convert_to_numpy=True,
            normalize_embeddings=True,
            batch_size=BATCH_SIZE,
            show_progress_bar=False
        ).astype("float32")   

        # ── Batch FAISS search ──
        scores, indices = index.search(query_embeddings, TOP_K)

        # ── Write results ──
        for i, item in enumerate(batch):
            retrieved_contexts = [
                {
                    "title": passage_texts[idx]["title"],
                    "text":  passage_texts[idx]["text"]
                }
                for idx in indices[i]
                if idx != -1  
            ]

            fout.write(json.dumps({
                "question":  item["question"],
                "reference": item["reference"],
                "contexts":  retrieved_contexts
            }, ensure_ascii=False) + "\n")


total questions: 65370


In [ ]:
import json

input_file = "nq_with_top20_contexts.jsonl"

output_file = "nq_questions_with_context.jsonl"

result = []

with open(input_file, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        question = item["question"]
        contexts = [p["text"] for p in item.get("contexts", [])]

        result.append({
            "question": question,
            "context": contexts
        })

with open(output_file, "w", encoding="utf-8") as f:
    for entry in result:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"Created {output_file} with {len(result)} questions.")

Created nq_questions_with_context.jsonl with 65370 questions.


## Hidden State Extraction for build preference data set

In [ ]:
from check_hidden_state import run

params = {
    "source": "data\Data_created\\nq_rag_test_convert2.jsonl",
    "type": "qa",
    "ra": "none",
    "model_path": "microsoft/phi-3-mini-4k-instruct",
    "batch_size": 1,
    "task": "nq",
    "max_new_tokens": 100,
    "hidden_states": True,               # extract hidden states
    "hidden_idx_mode": "first",       # pick which index of hidden states
    "need_layers": "mid",             # mid layer
    "gpu_device": "0",
    "weight_path": "training_data_mlp\weights\\best_model.pth",
    "hidden_prob_output_dir": "nq_rerank/hidden_prob_eval/"
}

# Run LLM to extract hidden states
run(params)

## Ectract Positive and negative samples

In [ ]:
import json
import sys
from collections import defaultdict
from utils.utils import read_json, write_jsonl

def transform_hidden_states_to_meta():
    try:
        input_file = "output_test/hidden_prob/hidden_state_by_llm.jsonl"
        meta_data = []
        
        with open(input_file, 'r') as f:
            for idx, line in enumerate(f):
                item = json.loads(line.strip())
                meta_data.append({
                    "question_id": idx,
                    "question": item.get("question", ""),
                    "context": item.get("Res", "")
                })
        
        write_jsonl(meta_data, "output_test/hidden_prob/nq_rerank_eval_meta.jsonl")
        
        return len(meta_data)
    
    except Exception:
        return 0

def generate_rerank_preference_dataset():
    try:
        meta_data = read_json("output_test/hidden_prob/nq_rerank_eval_meta.jsonl")
        prob_data = read_json("output_test/hidden_prob/hidden_prob_by_mlp.jsonl")
        
        if len(meta_data) != len(prob_data):
            return 0
        
        samples = []
        for i in range(len(meta_data)):
            _neg_prob, _pso_prob = prob_data[i]
            samples.append({
                "question_id": meta_data[i]["question_id"],
                "question": meta_data[i]["question"],
                "context": meta_data[i]["context"],
                "prob_no": _neg_prob,
                "prob_yes": _pso_prob,
                "has_answer_prob": _pso_prob
            })
        
        samples_sorted = sorted(samples, key=lambda x: x["has_answer_prob"], reverse=True)
        
        high_conf = [s for s in samples_sorted if s["has_answer_prob"] > 0.6]
        low_conf = [s for s in samples_sorted if s["has_answer_prob"] < 0.4]
        
        result = []
        
        for high_sample in high_conf:
            result.append({
                "query": high_sample["question"],
                "pos": [high_sample["context"]],
                "neg": [],
                "prompt": "Given a question, retrieve Wikipedia passages that answer the question."
            })
        
        for low_sample in low_conf:
            result.append({
                "query": low_sample["question"],
                "pos": [],
                "neg": [low_sample["context"]],
                "prompt": "Given a question, retrieve Wikipedia passages that answer the question."
            })
        
        write_jsonl(result, "output_test/reranking/rerank_preference.jsonl")
        
        return len(result)
    
    except Exception:
        return 0

if __name__ == "__main__":
    transformed_count = transform_hidden_states_to_meta()
    
    if transformed_count > 0:
        result_count = generate_rerank_preference_dataset()
        
        if result_count > 0:
            print(f"created positive and negative samples for {result_count} questions")

created positive and negative samples for 65370 questions


## ANSWER GENARTION PRECISION TRAIN FROM SCRATCH MODEL

In [ ]:
import os
import re
import gc
import json
import faiss
import pickle
import string
from tqdm import tqdm

import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# =========================================================
# CONFIG
# =========================================================
NQ_INPUT = r"data\nq-dev.jsonl"
FAISS_INDEX_PATH = r"passage_index.faiss"
MAPPING_PATH = r"passage_mapping.pkl"
SCRATCH_RERANKER_PATH = r"D:\git code\Reranker\model\scratch"

OUTPUT_DIR = "rag_accuracy_eval"
os.makedirs(OUTPUT_DIR, exist_ok=True)

EMBED_MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"

RETRIEVE_TOP_N = 20
QUERY_BATCH_SIZE = 64
RERANK_BATCH_SIZE = 16
MAX_LENGTH_RERANK = 512

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================================================
# HELPERS
# =========================================================
def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

# =========================================================
# LOAD DATA
# =========================================================
def load_nq_data(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            d = json.loads(line)

            refs = d.get("reference", [])
            if isinstance(refs, str):
                refs = [refs]

            data.append({
                "question": d["question"],
                "reference": refs
            })
    return data

# =========================================================
# LOAD FAISS + PASSAGES
# =========================================================
def load_faiss_and_mapping(index_path, mapping_path):
    print("Loading FAISS index on CPU...")
    index = faiss.read_index(index_path)

    print("Loading passage mapping...")
    with open(mapping_path, "rb") as f:
        mapping = pickle.load(f)

    passages = mapping["passages"] 
    return index, passages

# =========================================================
# RETRIEVE FROM FAISS
# =========================================================
def retrieve_candidates(data, index, passages, embed_model):
    print("Retrieving candidate passages from FAISS...")

    retrieved_data = []

    for start in tqdm(range(0, len(data), QUERY_BATCH_SIZE), desc="Retrieval"):
        batch = data[start:start + QUERY_BATCH_SIZE]
        questions = [x["question"] for x in batch]

        query_emb = embed_model.encode(
            questions,
            convert_to_numpy=True,
            normalize_embeddings=True,
            batch_size=QUERY_BATCH_SIZE,
            show_progress_bar=False
        ).astype("float32")

        scores, indices = index.search(query_emb, RETRIEVE_TOP_N)

        for i, item in enumerate(batch):
            contexts = []
            for idx, score in zip(indices[i], scores[i]):
                if idx == -1:
                    continue

                p = passages[idx]
                contexts.append({
                    "title": p["title"],
                    "text": p["text"],
                    "retrieval_score": float(score)
                })

            retrieved_data.append({
                "question": item["question"],
                "reference": item["reference"],
                "contexts": contexts
            })

    return retrieved_data

# =========================================================
# SCRATCH RERANKER
# =========================================================
class ScratchReranker:
    def __init__(self, checkpoint_path):
        print("Loading scratch reranker from:", checkpoint_path)
        self.tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
        self.model = AutoModelForSequenceClassification.from_pretrained(checkpoint_path)
        self.model.to(DEVICE)
        self.model.eval()

    def score_passages(self, question, contexts):
        scores = []

        for start in range(0, len(contexts), RERANK_BATCH_SIZE):
            batch_contexts = contexts[start:start + RERANK_BATCH_SIZE]

            pairs = [
                (
                    question,
                    (ctx["title"] + " " + ctx["text"]).strip()
                )
                for ctx in batch_contexts
            ]

            encoded = self.tokenizer(
                [p[0] for p in pairs],
                [p[1] for p in pairs],
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH_RERANK,
                return_tensors="pt"
            )

            encoded = {k: v.to(DEVICE) for k, v in encoded.items()}

            with torch.no_grad():
                outputs = self.model(**encoded)
                logits = outputs.logits

                if logits.dim() == 2 and logits.size(1) == 1:
                    batch_scores = logits.squeeze(1)
                elif logits.dim() == 2 and logits.size(1) == 2:
                    batch_scores = logits[:, 1]
                else:
                    batch_scores = logits.squeeze(-1)

                scores.extend(batch_scores.detach().cpu().tolist())

        return scores

# =========================================================
# RERANK
# =========================================================
def rerank_all(retrieved_data, reranker):
    print("Reranking retrieved passages with scratch model...")

    reranked_data = []

    for item in tqdm(retrieved_data, desc="Reranking"):
        question = item["question"]
        reference = item["reference"]
        contexts = item["contexts"]

        if len(contexts) == 0:
            reranked_data.append({
                "question": question,
                "reference": reference,
                "reranked_contexts": []
            })
            continue

        scores = reranker.score_passages(question, contexts)

        ranked = []
        for ctx, score in zip(contexts, scores):
            new_ctx = dict(ctx)
            new_ctx["rerank_score"] = float(score)
            ranked.append(new_ctx)

        ranked.sort(key=lambda x: x["rerank_score"], reverse=True)

        reranked_data.append({
            "question": question,
            "reference": reference,
            "reranked_contexts": ranked
        })

    return reranked_data

# =========================================================
# SAVE JSON
# =========================================================
def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

# =========================================================
# MAIN
# =========================================================
def main():
    print("==========================================")
    print("Device:", DEVICE)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    print("==========================================")

    data = load_nq_data(NQ_INPUT)

    # limit to 3000 questions
    MAX_QUESTIONS = 100
    data = data[:MAX_QUESTIONS]

    print("Total questions (limited):", len(data))

    # 2. load FAISS + passages
    index, passages = load_faiss_and_mapping(FAISS_INDEX_PATH, MAPPING_PATH)

    # 3. load embedding model
    print("Loading embedding model...")
    embed_model = SentenceTransformer(EMBED_MODEL_NAME, device="cpu")

    # 4. retrieve candidates
    retrieved_data = retrieve_candidates(data, index, passages, embed_model)
    save_json(retrieved_data[:20], os.path.join(OUTPUT_DIR, "retrieved_preview.json"))

    # 5. free embedding model
    del embed_model
    cleanup_memory()

    # 6. load reranker
    reranker = ScratchReranker(SCRATCH_RERANKER_PATH)

    # 7. rerank
    reranked_data = rerank_all(retrieved_data, reranker)
    save_json(reranked_data[:20], os.path.join(OUTPUT_DIR, "reranked_preview.json"))

    # 8. save full reranked file for Colab
    full_reranked_path = os.path.join(OUTPUT_DIR, "reranked_full.json")
    save_json(reranked_data, full_reranked_path)

    # 9. cleanup
    del reranker
    cleanup_memory()

    print(full_reranked_path)

if __name__ == "__main__":
    main()

d:\git code\Reranker\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
Total questions (limited): 100
Loading FAISS index on CPU...
Loading passage mapping...
Loading embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4140.77it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Retrieving candidate passages from FAISS...


Retrieval: 100%|██████████| 2/2 [00:03<00:00,  1.58s/it]


Loading scratch reranker from: D:\git code\Reranker\model\scratch


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3570.59it/s]


Reranking retrieved passages with scratch model...


Reranking: 100%|██████████| 100/100 [00:19<00:00,  5.05it/s]



Done.
Saved full reranked file here:
rag_accuracy_eval\reranked_full.json

Upload that file to Colab and continue from there.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# =========================
# IMPORTS
# =========================
import os
import re
import json
import string
from tqdm import tqdm

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# =========================
# CONFIG
# =========================
RERANKED_INPUT = "/content/drive/MyDrive/Data//scratch/reranked_full.json"  
OUTPUT_DIR = "/content/drive/MyDrive/Data/rag_accuracy_eval_colab"
os.makedirs(OUTPUT_DIR, exist_ok=True)

GENERATOR_MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
RERANK_TOP_K_LIST = [1, 3]
MAX_NEW_TOKENS = 64
GENERATOR_MAX_INPUT_LEN = 5000

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA L4


In [ ]:
with open(RERANKED_INPUT, "r", encoding="utf-8") as f:
    reranked_data = json.load(f)

reranked_data = reranked_data

print("Total reranked samples (limited):", len(reranked_data))
print("Example keys:", reranked_data[0].keys())

Total reranked samples (limited): 3000
Example keys: dict_keys(['question', 'reference', 'reranked_contexts'])


In [4]:
# =========================
# NORMALIZATION
# =========================
def normalize_text(s: str) -> str:
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = " ".join(s.split())
    return s

def answer_is_correct(prediction: str, references) -> int:
    pred_norm = normalize_text(prediction)

    for ref in references:
        ref_norm = normalize_text(ref)
        if not ref_norm:
            continue
        if ref_norm == pred_norm or ref_norm in pred_norm:
            return 1

    return 0

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    GENERATOR_MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)

model.eval()
print("model loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

model loaded


In [6]:
# =========================
# GENERATION
# =========================
def build_prompt(question, contexts):
    docs = []
    for i, ctx in enumerate(contexts, start=1):
        docs.append(
            f"[Document {i}]\nTitle: {ctx['title']}\nText: {ctx['text']}"
        )

    docs_text = "\n\n".join(docs)

    prompt = f"""You are given a question and supporting documents.
Answer using only the provided documents.
Give a short fact-based answer.
If the exact answer is not clearly supported by the documents, say: Not enough information.

Question:
{question}

Documents:
{docs_text}

Answer:"""
    return prompt

def generate_answer(question, contexts):
    prompt = build_prompt(question, contexts)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=GENERATOR_MAX_INPUT_LEN
    )

    if torch.cuda.is_available():
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    answer = answer.split("\n")[0].strip()

    return answer if answer else "Not enough information"

In [7]:
# =========================
# SAVE JSON
# =========================
def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

In [10]:
# =========================
# EVALUATE
# =========================
from tqdm.auto import tqdm
import time

def evaluate_rag_system_accuracy(reranked_data):
    print("Evaluating RAG System Accuracy...\n")

    results = {}

    for top_k in RERANK_TOP_K_LIST:
        correct_count = 0
        total_count = 0

        for item in reranked_data:
            question = item["question"]
            references = item["reference"]
            selected_contexts = item["reranked_contexts"][:top_k]

            if len(selected_contexts) == 0:
                prediction = "Not enough information"
            else:
                prediction = generate_answer(question, selected_contexts)

            is_correct = answer_is_correct(prediction, references)

            correct_count += int(is_correct)
            total_count += 1

        precision = correct_count / total_count if total_count > 0 else 0.0

        results[top_k] = precision

    return results

In [ ]:
# =========================
# RUN EVALUATION
# =========================
results = evaluate_rag_system_accuracy(reranked_data)

save_json(results, os.path.join(OUTPUT_DIR, "rag_system_accuracy_summary.json"))

print("\n================ FINAL RESULTS ================\n")
for top_k in RERANK_TOP_K_LIST:
    print(f"Top-{top_k}")
    print(f"Precision of Generated Answers: {results[top_k]:.4f}")
    print("-" * 50)

Evaluating RAG System Accuracy...


================ FINAL RESULTS ================

Top-1
Precision of Generated Answers: 0.4390
--------------------------------------------------
Top-3
Precision of Generated Answers: 0.5610
--------------------------------------------------


## ANSWER GENERATION EVALUATION Tinybert model

In [ ]:
import os
import gc
import json
import faiss
import pickle
from tqdm import tqdm

import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# =========================================================
# CONFIG
# =========================================================
NQ_INPUT = r"data\nq-dev.jsonl"
FAISS_INDEX_PATH = r"passage_index.faiss"
MAPPING_PATH = r"passage_mapping.pkl"

# MS MARCO TinyBERT reranker
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-TinyBERT-L-2-v2"

OUTPUT_DIR = "rag_accuracy_eval_tinybert"
os.makedirs(OUTPUT_DIR, exist_ok=True)

EMBED_MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"

RETRIEVE_TOP_N = 20
QUERY_BATCH_SIZE = 64
RERANK_BATCH_SIZE = 16
MAX_LENGTH_RERANK = 512

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================================================
# HELPERS
# =========================================================
def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

# =========================================================
# LOAD DATA
# =========================================================
def load_nq_data(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            d = json.loads(line)

            refs = d.get("reference", [])
            if isinstance(refs, str):
                refs = [refs]

            data.append({
                "question": d["question"],
                "reference": refs
            })
    return data

# =========================================================
# LOAD FAISS + PASSAGES
# =========================================================
def load_faiss_and_mapping(index_path, mapping_path):
    print("Loading FAISS index on CPU...")
    index = faiss.read_index(index_path)

    print("Loading passage mapping...")
    with open(mapping_path, "rb") as f:
        mapping = pickle.load(f)

    passages = mapping["passages"]
    return index, passages

# =========================================================
# RETRIEVE FROM FAISS
# =========================================================
def retrieve_candidates(data, index, passages, embed_model):
    print("Retrieving candidate passages from FAISS...")

    retrieved_data = []

    for start in tqdm(range(0, len(data), QUERY_BATCH_SIZE), desc="Retrieval"):
        batch = data[start:start + QUERY_BATCH_SIZE]
        questions = [x["question"] for x in batch]

        query_emb = embed_model.encode(
            questions,
            convert_to_numpy=True,
            normalize_embeddings=True,
            batch_size=QUERY_BATCH_SIZE,
            show_progress_bar=False
        ).astype("float32")

        scores, indices = index.search(query_emb, RETRIEVE_TOP_N)

        for i, item in enumerate(batch):
            contexts = []
            for idx, score in zip(indices[i], scores[i]):
                if idx == -1:
                    continue

                p = passages[idx]
                contexts.append({
                    "title": p["title"],
                    "text": p["text"],
                    "retrieval_score": float(score)
                })

            retrieved_data.append({
                "question": item["question"],
                "reference": item["reference"],
                "contexts": contexts
            })

    return retrieved_data

# =========================================================
# MS MARCO TINYBERT RERANKER
# =========================================================
class MSMARCOTinyBERTReranker:
    def __init__(self, model_name):
        print("Loading reranker:", model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.model.to(DEVICE)
        self.model.eval()

    def score_passages(self, question, contexts):
        scores = []

        for start in range(0, len(contexts), RERANK_BATCH_SIZE):
            batch_contexts = contexts[start:start + RERANK_BATCH_SIZE]

            queries = [question] * len(batch_contexts)
            docs = [
                (ctx["title"] + " " + ctx["text"]).strip()
                for ctx in batch_contexts
            ]

            encoded = self.tokenizer(
                queries,
                docs,
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH_RERANK,
                return_tensors="pt"
            )

            encoded = {k: v.to(DEVICE) for k, v in encoded.items()}

            with torch.no_grad():
                outputs = self.model(**encoded)
                logits = outputs.logits

                if logits.dim() == 2 and logits.size(1) == 1:
                    batch_scores = logits.squeeze(1)
                elif logits.dim() == 2 and logits.size(1) == 2:
                    batch_scores = logits[:, 1]
                else:
                    batch_scores = logits.squeeze(-1)

                scores.extend(batch_scores.detach().cpu().tolist())

        return scores

# =========================================================
# RERANK
# =========================================================
def rerank_all(retrieved_data, reranker):
    print("Reranking retrieved passages with MS MARCO TinyBERT...")

    reranked_data = []

    for item in tqdm(retrieved_data, desc="Reranking"):
        question = item["question"]
        reference = item["reference"]
        contexts = item["contexts"]

        if len(contexts) == 0:
            reranked_data.append({
                "question": question,
                "reference": reference,
                "reranked_contexts": []
            })
            continue

        scores = reranker.score_passages(question, contexts)

        ranked = []
        for ctx, score in zip(contexts, scores):
            new_ctx = dict(ctx)
            new_ctx["rerank_score"] = float(score)
            ranked.append(new_ctx)

        ranked.sort(key=lambda x: x["rerank_score"], reverse=True)

        reranked_data.append({
            "question": question,
            "reference": reference,
            "reranked_contexts": ranked
        })

    return reranked_data

# =========================================================
# SAVE JSON
# =========================================================
def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

# =========================================================
# MAIN
# =========================================================
def main():
    print("==========================================")
    print("Device:", DEVICE)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    print("==========================================")

    data = load_nq_data(NQ_INPUT)

    # limit to 3000 questions
    MAX_QUESTIONS = 3000
    data = data[:MAX_QUESTIONS]

    print("Total questions (limited):", len(data))

    # 1. load FAISS + passages
    index, passages = load_faiss_and_mapping(FAISS_INDEX_PATH, MAPPING_PATH)

    # 2. load embedding model
    print("Loading embedding model...")
    embed_model = SentenceTransformer(EMBED_MODEL_NAME, device="cpu")

    # 3. retrieve candidates
    retrieved_data = retrieve_candidates(data, index, passages, embed_model)
    save_json(retrieved_data[:20], os.path.join(OUTPUT_DIR, "retrieved_preview.json"))

    # 4. free embedding model
    del embed_model
    cleanup_memory()

    # 5. load MS MARCO TinyBERT reranker
    reranker = MSMARCOTinyBERTReranker(RERANKER_MODEL_NAME)

    # 6. rerank
    reranked_data = rerank_all(retrieved_data, reranker)
    save_json(reranked_data[:20], os.path.join(OUTPUT_DIR, "reranked_preview.json"))

    # 7. save full reranked file for Colab
    full_reranked_path = os.path.join(OUTPUT_DIR, "reranked_full_tinybert.json")
    save_json(reranked_data, full_reranked_path)

    # 8. cleanup
    del reranker
    cleanup_memory()


if __name__ == "__main__":
    main()

d:\git code\Reranker\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
Total questions (limited): 3000
Loading FAISS index on CPU...
Loading passage mapping...
Loading embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2080.33it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Retrieving candidate passages from FAISS...


Retrieval: 100%|██████████| 47/47 [02:04<00:00,  2.66s/it]


Loading reranker: cross-encoder/ms-marco-TinyBERT-L-2-v2


d:\git code\Reranker\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\models--cross-encoder--ms-marco-TinyBERT-L-2-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 41/41 [00:00<00:00, 2771.77it/s]
BertForSequenceClassification LOAD RE

Reranking retrieved passages with MS MARCO TinyBERT...


Reranking: 100%|██████████| 3000/3000 [00:48<00:00, 62.34it/s]



Done.
Saved full reranked file here:
rag_accuracy_eval_tinybert\reranked_full_tinybert.json

Upload that file to Colab and continue from there.


In [ ]:
import os
import re
import json
import string
from tqdm import tqdm

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


# =========================
# CONFIG
# =========================
RERANKED_INPUT = "/content/drive/MyDrive/Data/reranked_full_tinybert.json"   
OUTPUT_DIR = "/content/drive/MyDrive/Data/rag_accuracy_eval_colab_tinybert"
os.makedirs(OUTPUT_DIR, exist_ok=True)

GENERATOR_MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
RERANK_TOP_K_LIST = [1, 3]
MAX_NEW_TOKENS = 64
GENERATOR_MAX_INPUT_LEN = 3500

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

with open(RERANKED_INPUT, "r", encoding="utf-8") as f:
    reranked_data = json.load(f)

reranked_data = reranked_data

# =========================
# NORMALIZATION
# =========================
def normalize_text(s: str) -> str:
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = " ".join(s.split())
    return s

def answer_is_correct(prediction: str, references) -> int:
    pred_norm = normalize_text(prediction)

    for ref in references:
        ref_norm = normalize_text(ref)
        if not ref_norm:
            continue
        if ref_norm == pred_norm or ref_norm in pred_norm:
            return 1

    return 0

GENERATOR_MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    GENERATOR_MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)

model.eval()
print("model loaded")

# =========================
# GENERATION
# =========================
def build_prompt(question, contexts):
    docs = []
    for i, ctx in enumerate(contexts, start=1):
        docs.append(
            f"[Document {i}]\nTitle: {ctx['title']}\nText: {ctx['text']}"
        )

    docs_text = "\n\n".join(docs)

    prompt = f"""You are given a question and supporting documents.
Answer the question briefly using only the provided documents.
If the answer is not in the documents, say: Not enough information.

Question:
{question}

Documents:
{docs_text}

Answer:"""
    return prompt

def generate_answer(question, contexts):
    prompt = build_prompt(question, contexts)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=GENERATOR_MAX_INPUT_LEN
    )

    if torch.cuda.is_available():
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    answer = answer.split("\n")[0].strip()

    return answer if answer else "Not enough information"

# =========================
# EVALUATE
# =========================

def evaluate_rag_system_accuracy(reranked_data):
    print("Evaluating RAG System Accuracy...\n")

    results = {}

    for top_k in RERANK_TOP_K_LIST:
        correct_count = 0
        total_count = 0

        for item in reranked_data:
            question = item["question"]
            references = item["reference"]
            selected_contexts = item["reranked_contexts"][:top_k]

            if len(selected_contexts) == 0:
                prediction = "Not enough information"
            else:
                prediction = generate_answer(question, selected_contexts)

            is_correct = answer_is_correct(prediction, references)

            correct_count += int(is_correct)
            total_count += 1

        precision = correct_count / total_count if total_count > 0 else 0.0

        results[top_k] = precision

    return results

In [13]:
# =========================
# RUN EVALUATION
# =========================
results = evaluate_rag_system_accuracy(reranked_data)

save_json(results, os.path.join(OUTPUT_DIR, "rag_system_accuracy_summary.json"))

print("\n================ FINAL RESULTS ================\n")
for top_k in RERANK_TOP_K_LIST:
    print(f"Top-{top_k}")
    print(f"Precision of Generated Answers: {results[top_k]:.4f}")
    print("-" * 50)

Evaluating RAG System Accuracy...


================ FINAL RESULTS ================

Top-1
Precision of Generated Answers: 0.4220
--------------------------------------------------
Top-3
Precision of Generated Answers: 0.5140
--------------------------------------------------


## ANSWER GENERATION EVALUATION Tinybert finetuned model

In [ ]:
import os
import gc
import json
import faiss
import pickle
from tqdm import tqdm

import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# =========================================================
# CONFIG
# =========================================================
NQ_INPUT = r"data\nq-dev.jsonl"
FAISS_INDEX_PATH = r"passage_index.faiss"
MAPPING_PATH = r"passage_mapping.pkl"

# MS MARCO TinyBERT reranker
RERANKER_MODEL_NAME = "tinybert_msmarco_finetuned"

OUTPUT_DIR = "rag_accuracy_eval_tinybert_ft"
os.makedirs(OUTPUT_DIR, exist_ok=True)

EMBED_MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"

RETRIEVE_TOP_N = 20
QUERY_BATCH_SIZE = 64
RERANK_BATCH_SIZE = 16
MAX_LENGTH_RERANK = 512

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================================================
# HELPERS
# =========================================================
def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

# =========================================================
# LOAD DATA
# =========================================================
def load_nq_data(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            d = json.loads(line)

            refs = d.get("reference", [])
            if isinstance(refs, str):
                refs = [refs]

            data.append({
                "question": d["question"],
                "reference": refs
            })
    return data

# =========================================================
# LOAD FAISS + PASSAGES
# =========================================================
def load_faiss_and_mapping(index_path, mapping_path):
    print("Loading FAISS index on CPU...")
    index = faiss.read_index(index_path)

    print("Loading passage mapping...")
    with open(mapping_path, "rb") as f:
        mapping = pickle.load(f)

    passages = mapping["passages"]
    return index, passages

# =========================================================
# RETRIEVE FROM FAISS
# =========================================================
def retrieve_candidates(data, index, passages, embed_model):
    print("Retrieving candidate passages from FAISS...")

    retrieved_data = []

    for start in tqdm(range(0, len(data), QUERY_BATCH_SIZE), desc="Retrieval"):
        batch = data[start:start + QUERY_BATCH_SIZE]
        questions = [x["question"] for x in batch]

        query_emb = embed_model.encode(
            questions,
            convert_to_numpy=True,
            normalize_embeddings=True,
            batch_size=QUERY_BATCH_SIZE,
            show_progress_bar=False
        ).astype("float32")

        scores, indices = index.search(query_emb, RETRIEVE_TOP_N)

        for i, item in enumerate(batch):
            contexts = []
            for idx, score in zip(indices[i], scores[i]):
                if idx == -1:
                    continue

                p = passages[idx]
                contexts.append({
                    "title": p["title"],
                    "text": p["text"],
                    "retrieval_score": float(score)
                })

            retrieved_data.append({
                "question": item["question"],
                "reference": item["reference"],
                "contexts": contexts
            })

    return retrieved_data

# =========================================================
# MS MARCO TINYBERT RERANKER
# =========================================================
class MSMARCOTinyBERTReranker:
    def __init__(self, model_name):
        print("Loading reranker:", model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.model.to(DEVICE)
        self.model.eval()

    def score_passages(self, question, contexts):
        scores = []

        for start in range(0, len(contexts), RERANK_BATCH_SIZE):
            batch_contexts = contexts[start:start + RERANK_BATCH_SIZE]

            queries = [question] * len(batch_contexts)
            docs = [
                (ctx["title"] + " " + ctx["text"]).strip()
                for ctx in batch_contexts
            ]

            encoded = self.tokenizer(
                queries,
                docs,
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH_RERANK,
                return_tensors="pt"
            )

            encoded = {k: v.to(DEVICE) for k, v in encoded.items()}

            with torch.no_grad():
                outputs = self.model(**encoded)
                logits = outputs.logits

                if logits.dim() == 2 and logits.size(1) == 1:
                    batch_scores = logits.squeeze(1)
                elif logits.dim() == 2 and logits.size(1) == 2:
                    batch_scores = logits[:, 1]
                else:
                    batch_scores = logits.squeeze(-1)

                scores.extend(batch_scores.detach().cpu().tolist())

        return scores

# =========================================================
# RERANK
# =========================================================
def rerank_all(retrieved_data, reranker):
    print("Reranking retrieved passages with MS MARCO TinyBERT...")

    reranked_data = []

    for item in tqdm(retrieved_data, desc="Reranking"):
        question = item["question"]
        reference = item["reference"]
        contexts = item["contexts"]

        if len(contexts) == 0:
            reranked_data.append({
                "question": question,
                "reference": reference,
                "reranked_contexts": []
            })
            continue

        scores = reranker.score_passages(question, contexts)

        ranked = []
        for ctx, score in zip(contexts, scores):
            new_ctx = dict(ctx)
            new_ctx["rerank_score"] = float(score)
            ranked.append(new_ctx)

        ranked.sort(key=lambda x: x["rerank_score"], reverse=True)

        reranked_data.append({
            "question": question,
            "reference": reference,
            "reranked_contexts": ranked
        })

    return reranked_data

# =========================================================
# SAVE JSON
# =========================================================
def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

# =========================================================
# MAIN
# =========================================================
def main():
    print("==========================================")
    print("Device:", DEVICE)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    print("==========================================")

    data = load_nq_data(NQ_INPUT)

    # limit to 3000 questions
    MAX_QUESTIONS = 3000
    data = data[:MAX_QUESTIONS]

    print("Total questions (limited):", len(data))

    # 1. load FAISS + passages
    index, passages = load_faiss_and_mapping(FAISS_INDEX_PATH, MAPPING_PATH)

    # 2. load embedding model
    print("Loading embedding model...")
    embed_model = SentenceTransformer(EMBED_MODEL_NAME, device="cpu")

    # 3. retrieve candidates
    retrieved_data = retrieve_candidates(data, index, passages, embed_model)
    save_json(retrieved_data[:20], os.path.join(OUTPUT_DIR, "retrieved_preview.json"))

    # 4. free embedding model
    del embed_model
    cleanup_memory()

    # 5. load MS MARCO TinyBERT reranker
    reranker = MSMARCOTinyBERTReranker(RERANKER_MODEL_NAME)

    # 6. rerank
    reranked_data = rerank_all(retrieved_data, reranker)
    save_json(reranked_data[:20], os.path.join(OUTPUT_DIR, "reranked_preview.json"))

    # 7. save full reranked file for Colab
    full_reranked_path = os.path.join(OUTPUT_DIR, "reranked_full_tinybert.json")
    save_json(reranked_data, full_reranked_path)

    # 8. cleanup
    del reranker
    cleanup_memory()

    print("\nDone.")
    print("Saved full reranked file here:")
    print(full_reranked_path)
    print("\nUpload that file to Colab and continue from there.")

if __name__ == "__main__":
    main()

d:\git code\Reranker\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
Total questions (limited): 3000
Loading FAISS index on CPU...
Loading passage mapping...
Loading embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4914.49it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Retrieving candidate passages from FAISS...


Retrieval: 100%|██████████| 47/47 [00:55<00:00,  1.18s/it]


Loading reranker: tinybert_msmarco_finetuned


Loading weights: 100%|██████████| 41/41 [00:00<?, ?it/s]


Reranking retrieved passages with MS MARCO TinyBERT...


Reranking: 100%|██████████| 3000/3000 [00:13<00:00, 216.97it/s]



Done.
Saved full reranked file here:
rag_accuracy_eval_tinybert_ft\reranked_full_tinybert.json

Upload that file to Colab and continue from there.


In [ ]:
# =========================
# IMPORTS
# =========================
import os
import re
import json
import string
from tqdm import tqdm

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


# =========================
# CONFIG
# =========================
RERANKED_INPUT = "/content/drive/MyDrive/Data/reranked_full_tinybert_ft.json"   
OUTPUT_DIR = "/content/drive/MyDrive/Data/rag_accuracy_eval_colab_tinybertft"
os.makedirs(OUTPUT_DIR, exist_ok=True)

GENERATOR_MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
RERANK_TOP_K_LIST = [1, 3]
MAX_NEW_TOKENS = 64
GENERATOR_MAX_INPUT_LEN = 3500

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

with open(RERANKED_INPUT, "r", encoding="utf-8") as f:
    reranked_data = json.load(f)

reranked_data = reranked_data


# =========================
# NORMALIZATION
# =========================
def normalize_text(s: str) -> str:
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = " ".join(s.split())
    return s

def answer_is_correct(prediction: str, references) -> int:
    pred_norm = normalize_text(prediction)

    for ref in references:
        ref_norm = normalize_text(ref)
        if not ref_norm:
            continue
        if ref_norm == pred_norm or ref_norm in pred_norm:
            return 1

    return 0

# GENERATOR_MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

# tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL_NAME)

# if tokenizer.pad_token is None:
#     tokenizer.pad_token = tokenizer.eos_token

# model = AutoModelForCausalLM.from_pretrained(
#     GENERATOR_MODEL_NAME,
#     torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
#     device_map="auto" if torch.cuda.is_available() else None,
# )

# model.eval()
# print("model loaded")

# =========================
# GENERATION
# =========================
def build_prompt(question, contexts):
    docs = []
    for i, ctx in enumerate(contexts, start=1):
        docs.append(
            f"[Document {i}]\nTitle: {ctx['title']}\nText: {ctx['text']}"
        )

    docs_text = "\n\n".join(docs)

    prompt = f"""You are given a question and supporting documents.
Answer the question briefly using only the provided documents.
If the answer is not in the documents, say: Not enough information.

Question:
{question}

Documents:
{docs_text}

Answer:"""
    return prompt

def generate_answer(question, contexts):
    prompt = build_prompt(question, contexts)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=GENERATOR_MAX_INPUT_LEN
    )

    if torch.cuda.is_available():
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    answer = answer.split("\n")[0].strip()

    return answer if answer else "Not enough information"

# =========================
# EVALUATE
# =========================
def evaluate_rag_system_accuracy(reranked_data):
    print("Evaluating RAG System Accuracy...\n")

    results = {}

    for top_k in RERANK_TOP_K_LIST:
        correct_count = 0
        total_count = 0

        for item in reranked_data:
            question = item["question"]
            references = item["reference"]
            selected_contexts = item["reranked_contexts"][:top_k]

            if len(selected_contexts) == 0:
                prediction = "Not enough information"
            else:
                prediction = generate_answer(question, selected_contexts)

            is_correct = answer_is_correct(prediction, references)

            correct_count += int(is_correct)
            total_count += 1

        precision = correct_count / total_count if total_count > 0 else 0.0

        results[top_k] = precision

    return results

In [15]:
# =========================
# RUN EVALUATION
# =========================
results = evaluate_rag_system_accuracy(reranked_data)

save_json(results, os.path.join(OUTPUT_DIR, "rag_system_accuracy_summary.json"))

print("\n================ FINAL RESULTS ================\n")
for top_k in RERANK_TOP_K_LIST:
    print(f"Top-{top_k}")
    print(f"Precision of Generated Answers: {results[top_k]:.4f}")
    print("-" * 50)

Evaluating RAG System Accuracy...


================ FINAL RESULTS ================

Top-1
Precision of Generated Answers: 0.4360
--------------------------------------------------
Top-3
Precision of Generated Answers: 0.5270
--------------------------------------------------


## ANSWER GENERATION EVALUATION bge model

In [ ]:
import os
import gc
import json
import faiss
import pickle
from tqdm import tqdm

from sentence_transformers import SentenceTransformer

# =========================================================
# CONFIG
# =========================================================
NQ_INPUT = r"data\nq-dev.jsonl"
FAISS_INDEX_PATH = r"passage_index.faiss"
MAPPING_PATH = r"passage_mapping.pkl"

OUTPUT_DIR = "rag_retrieval_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

EMBED_MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"

RETRIEVE_TOP_N = 20
QUERY_BATCH_SIZE = 64
RERANK_BATCH_SIZE = 16
MAX_LENGTH_RERANK = 512

# =========================================================
# HELPERS
# =========================================================
def cleanup_memory():
    gc.collect()

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

# =========================================================
# LOAD DATA
# =========================================================
def load_nq_data(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            d = json.loads(line)

            refs = d.get("reference", [])
            if isinstance(refs, str):
                refs = [refs]

            data.append({
                "question": d["question"],
                "reference": refs
            })
    return data

# =========================================================
# LOAD FAISS + PASSAGES
# =========================================================
def load_faiss_and_mapping(index_path, mapping_path):
    print("Loading FAISS index on CPU...")
    index = faiss.read_index(index_path)

    print("Loading passage mapping...")
    with open(mapping_path, "rb") as f:
        mapping = pickle.load(f)

    passages = mapping["passages"]
    return index, passages

# =========================================================
# RETRIEVE
# =========================================================
def retrieve_candidates(data, index, passages, embed_model):
    print("Retrieving candidate passages from FAISS...")

    retrieved_data = []

    for start in tqdm(range(0, len(data), QUERY_BATCH_SIZE), desc="Retrieval"):
        batch = data[start:start + QUERY_BATCH_SIZE]
        questions = [x["question"] for x in batch]

        query_emb = embed_model.encode(
            questions,
            convert_to_numpy=True,
            normalize_embeddings=True,
            batch_size=QUERY_BATCH_SIZE,
            show_progress_bar=False
        ).astype("float32")

        scores, indices = index.search(query_emb, RETRIEVE_TOP_N)

        for i, item in enumerate(batch):
            contexts = []
            for idx, score in zip(indices[i], scores[i]):
                if idx == -1:
                    continue

                p = passages[idx]
                contexts.append({
                    "title": p["title"],
                    "text": p["text"],
                    "retrieval_score": float(score)
                })

            retrieved_data.append({
                "question": item["question"],
                "reference": item["reference"],
                "contexts": contexts
            })

    return retrieved_data

# =========================================================
# MAIN
# =========================================================
def main():
    print("==========================================")
    print("Stage 1: Retrieval only")
    print("==========================================")

    # 1. load data
    data = load_nq_data(NQ_INPUT)
    data = data[:MAX_QUESTIONS]
    print("Total questions (limited):", len(data))

    # 2. load FAISS + passages
    index, passages = load_faiss_and_mapping(FAISS_INDEX_PATH, MAPPING_PATH)

    # 3. load embedding model
    print("Loading embedding model...")
    embed_model = SentenceTransformer(EMBED_MODEL_NAME, device="cpu")

    # 4. retrieve
    retrieved_data = retrieve_candidates(data, index, passages, embed_model)

    # 5. save outputs
    save_json(
        retrieved_data[:20],
        os.path.join(OUTPUT_DIR, "retrieved_preview.json")
    )

    retrieved_full_path = os.path.join(OUTPUT_DIR, "retrieved_full.json")
    save_json(retrieved_data, retrieved_full_path)

    # 6. cleanup
    del embed_model
    cleanup_memory()

if __name__ == "__main__":
    main()

d:\git code\Reranker\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Stage 1: Retrieval only
Total questions (limited): 3000
Loading FAISS index on CPU...
Loading passage mapping...
Loading embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3952.73it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Retrieving candidate passages from FAISS...


Retrieval: 100%|██████████| 47/47 [00:54<00:00,  1.16s/it]


In [1]:
import os
import re
import gc
import json
import time
import string
from tqdm.auto import tqdm

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM, AutoConfig

# =========================
# CONFIG
# =========================
RETRIEVED_INPUT = "/content/drive/MyDrive/Data/retrieved_full.json"  
OUTPUT_DIR = "/content/drive/MyDrive/Data/rag_eval_colab"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"

GENERATOR_MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

RERANK_TOP_K_LIST = [1, 3]
RERANK_BATCH_SIZE = 32
MAX_LENGTH_RERANK = 256
MAX_NEW_TOKENS = 64
GENERATOR_MAX_INPUT_LEN = 3500


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# =========================
# HELPERS
# =========================
def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def normalize_text(s: str) -> str:
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = " ".join(s.split())
    return s

def answer_is_correct(prediction: str, references) -> int:
    pred_norm = normalize_text(prediction)

    for ref in references:
        ref_norm = normalize_text(ref)
        if not ref_norm:
            continue
        if ref_norm == pred_norm or ref_norm in pred_norm:
            return 1

    return 0

# =========================
# LOAD RETRIEVED DATA
# =========================
with open(RETRIEVED_INPUT, "r", encoding="utf-8") as f:
    retrieved_data = json.load(f)

# =========================
# RERANKER
# =========================
class Reranker:
    def __init__(self, model_name):
        print("Loading reranker:", model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.model.to(DEVICE)
        self.model.eval()

    def score_passages(self, question, contexts):
        scores = []
        for start in range(0, len(contexts), RERANK_BATCH_SIZE):
            batch_contexts = contexts[start:start + RERANK_BATCH_SIZE]

            queries = [question] * len(batch_contexts)
            docs = [(ctx["title"] + " " + ctx["text"]).strip() for ctx in batch_contexts]

            encoded = self.tokenizer(
                queries,
                docs,
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH_RERANK,
                return_tensors="pt"
            )
            encoded = {k: v.to(DEVICE) for k, v in encoded.items()}

            with torch.no_grad():
                outputs = self.model(**encoded)
                logits = outputs.logits

                if logits.dim() == 2 and logits.size(1) == 1:
                    batch_scores = logits.squeeze(1)
                elif logits.dim() == 2 and logits.size(1) == 2:
                    batch_scores = logits[:, 1]
                else:
                    batch_scores = logits.squeeze(-1)

                scores.extend(batch_scores.detach().cpu().tolist())
        return scores

def rerank_all(retrieved_data, reranker):
    print("Reranking retrieved passages...")
    reranked_data = []

    for item in tqdm(retrieved_data, desc="Reranking"):
        question = item["question"]
        reference = item["reference"]
        contexts = item["contexts"]

        if not contexts:
            reranked_data.append({
                "question": question,
                "reference": reference,
                "reranked_contexts": []
            })
            continue

        scores = reranker.score_passages(question, contexts)

        ranked = []
        for ctx, score in zip(contexts, scores):
            new_ctx = dict(ctx)
            new_ctx["rerank_score"] = float(score)
            ranked.append(new_ctx)

        ranked.sort(key=lambda x: x["rerank_score"], reverse=True)

        reranked_data.append({
            "question": question,
            "reference": reference,
            "reranked_contexts": ranked
        })

    return reranked_data

reranker = Reranker(RERANKER_MODEL_NAME)
reranked_data = rerank_all(retrieved_data, reranker)

save_json(reranked_data[:20], os.path.join(OUTPUT_DIR, "reranked_preview.json"))
save_json(reranked_data, os.path.join(OUTPUT_DIR, "reranked_full.json"))

del reranker
cleanup_memory()
print("Reranking complete.")

# =========================
# LOAD PHI-3 SAFELY
# =========================
print("Loading generator:", GENERATOR_MODEL_NAME)

# Load config first and patch rope_scaling if needed
gen_config = AutoConfig.from_pretrained(GENERATOR_MODEL_NAME, trust_remote_code=True)

if hasattr(gen_config, "rope_scaling") and gen_config.rope_scaling is not None:
    rs = gen_config.rope_scaling
    if isinstance(rs, dict):
        if "type" not in rs and "rope_type" in rs:
            rs["type"] = rs["rope_type"]
        elif "type" not in rs:
            rs["type"] = "longrope"
        gen_config.rope_scaling = rs

gen_tokenizer = AutoTokenizer.from_pretrained(
    GENERATOR_MODEL_NAME,
    trust_remote_code=True
)

if gen_tokenizer.pad_token is None:
    gen_tokenizer.pad_token = gen_tokenizer.eos_token

generator = AutoModelForCausalLM.from_pretrained(
    GENERATOR_MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)

generator.eval()
print("Generator loaded.")

# =========================
# GENERATION
# =========================
def build_prompt(question, contexts):
    docs = []
    for i, ctx in enumerate(contexts, start=1):
        docs.append(f"[Document {i}]\nTitle: {ctx['title']}\nText: {ctx['text']}")
    docs_text = "\n\n".join(docs)

    return f"""You are given a question and supporting documents.
Answer the question briefly using only the provided documents.
If the answer is not in the documents, say: Not enough information.

Question:
{question}

Documents:
{docs_text}

Answer:"""

def generate_answer(question, contexts):
    prompt = build_prompt(question, contexts)

    inputs = gen_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=GENERATOR_MAX_INPUT_LEN
    )

    if torch.cuda.is_available():
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

    with torch.no_grad():
        outputs = generator.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=gen_tokenizer.eos_token_id,
            eos_token_id=gen_tokenizer.eos_token_id
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    answer = gen_tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    answer = answer.split("\n")[0].strip()

    return answer if answer else "Not enough information"

# =========================
# EVALUATE
# =========================
def evaluate_rag_system_accuracy(reranked_data):
    print("Evaluating RAG System Accuracy...\n")

    results = {}

    for top_k in RERANK_TOP_K_LIST:
        correct_count = 0
        total_count = 0

        for item in reranked_data:
            question = item["question"]
            references = item["reference"]
            selected_contexts = item["reranked_contexts"][:top_k]

            if len(selected_contexts) == 0:
                prediction = "Not enough information"
            else:
                prediction = generate_answer(question, selected_contexts)

            is_correct = answer_is_correct(prediction, references)

            correct_count += int(is_correct)
            total_count += 1

        precision = correct_count / total_count if total_count > 0 else 0.0

        results[top_k] = precision

    return results

Loading reranker: BAAI/bge-reranker-v2-m3


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Generator loaded.


In [2]:
# =========================
# RUN EVALUATION
# =========================
results = evaluate_rag_system_accuracy(reranked_data)

save_json(results, os.path.join(OUTPUT_DIR, "rag_system_accuracy_summary.json"))

print("\n================ FINAL RESULTS ================\n")
for top_k in RERANK_TOP_K_LIST:
    print(f"Top-{top_k}")
    print(f"Precision of Generated Answers: {results[top_k]:.4f}")
    print("-" * 50)

Evaluating RAG System Accuracy...


================ FINAL RESULTS ================

Top-1
Precision of Generated Answers: 0.5260
--------------------------------------------------
Top-3
Precision of Generated Answers: 0.6140
--------------------------------------------------
